In [1]:
# Download processed dataset from shared Drive
import gdown, zipfile, os

if not os.path.exists('dataset/processed'):
    FILE_ID = '1a7kCwqP2Vhlv43eep0ODdzGVR6v2ouP6'
    gdown.download(id=FILE_ID, output='processed.zip', quiet=False)
    with zipfile.ZipFile('processed.zip', 'r') as z:
        z.extractall('dataset/')
    os.remove('processed.zip')
else:
    print('Already downloaded.')

Downloading...
From (original): https://drive.google.com/uc?id=1a7kCwqP2Vhlv43eep0ODdzGVR6v2ouP6
From (redirected): https://drive.google.com/uc?id=1a7kCwqP2Vhlv43eep0ODdzGVR6v2ouP6&confirm=t&uuid=3814eb0a-8fed-4904-a595-0ade9207f1b6
To: /content/processed.zip
100%|██████████| 540M/540M [00:02<00:00, 200MB/s]


In [2]:
# Load shared splits and ID mappings
import pandas as pd
import numpy as np
import pickle

train_df = pd.read_csv('dataset/processed/train.csv')
val_df   = pd.read_csv('dataset/processed/val.csv')
test_df  = pd.read_csv('dataset/processed/test.csv')
games_df = pd.read_csv('dataset/processed/games_cleaned.csv')

with open('dataset/processed/id_maps.pkl', 'rb') as f:
    maps = pickle.load(f)

n_users = len(maps['user2idx'])
n_items = len(maps['item2idx'])

print(f'n_users: {n_users:,} | n_items: {n_items:,}')
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

n_users: 272,184 | n_items: 21,922
Train: 18,151,978 | Val: 272,184 | Test: 272,184


# Implicit Item-based CF

In [3]:
!pip install implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=933264 sha256=9724220c170bd5c985a7aae410fc289e36c1469a9abe388b428665595e9deedc
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


In [4]:
from scipy.sparse import csr_matrix
from implicit.nearest_neighbours import CosineRecommender
from tqdm.auto import tqdm

def create_sparse_matrix(df, n_users, n_items):
    rows = df['user_idx'].values
    cols = df['item_idx'].values
    data = np.ones(len(rows))
    return csr_matrix((data, (rows, cols)), shape=(n_users, n_items)).tocsr()

train_matrix = create_sparse_matrix(train_df, n_users, n_items)

In [5]:
print('Train matrix shape:', train_matrix.shape)

Train matrix shape: (272184, 21922)


In [6]:
item_cf_model = CosineRecommender(K=40)
item_cf_model.fit(train_matrix)

/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.16886544227600098 seconds
  warnings.warn(


  0%|          | 0/21922 [00:00<?, ?it/s]

In [7]:
print('Similarity matrix shape:', item_cf_model.similarity.shape)

Similarity matrix shape: (21922, 21922)


# Evaluation

In [9]:
# Shared evaluation function

# moved outside of function to speed up evaluation
seen = train_df.groupby('user_idx')['item_idx'].apply(set).to_dict()
val_seen = val_df.groupby('user_idx')['item_idx'].apply(set).to_dict()

def evaluate(score_series, train_df, test_df, n_negatives=99, K=10, seed=42):
    rng = np.random.default_rng(seed)
    all_items = np.array(list(maps['item2idx'].values()))
    hits, ndcgs, mrrs = [], [], []

    for _, row in test_df.iterrows():
        uid      = int(row['user_idx'])
        pos_item = int(row['item_idx'])

        user_seen  = seen.get(uid, set()) | val_seen.get(uid, set()) | {pos_item}
        candidates = np.setdiff1d(all_items, list(user_seen))
        neg_items  = rng.choice(candidates, size=n_negatives, replace=False)

        items_to_rank = np.append(neg_items, pos_item)
        scores = score_series.reindex(items_to_rank).fillna(0).values
        ranked = items_to_rank[np.argsort(-scores)]

        pos_rank = np.where(ranked == pos_item)[0][0] + 1

        hits.append(1 if pos_rank <= K else 0)
        ndcgs.append(np.log(2) / np.log(pos_rank + 1) if pos_rank <= K else 0)
        mrrs.append(1 / pos_rank if pos_rank <= K else 0)

    return {
        f'HR@{K}':   round(np.mean(hits), 4),
        f'NDCG@{K}': round(np.mean(ndcgs), 4),
        f'MRR@{K}':  round(np.mean(mrrs), 4),
    }

# Results

In [10]:
def generate_series_for_user(model, user_id, train_matrix, n_items):
    user_vector = train_matrix[user_id]

    # User vector * similarty matrix
    scores = user_vector.dot(model.similarity).toarray().flatten()
    return pd.Series(scores, index=np.arange(n_items))

In [11]:
from tqdm import tqdm
all_scores = {}
np.random.seed(42)
for uid in tqdm(test_df['user_idx'].unique()):
  user_series = generate_series_for_user(item_cf_model, uid, train_matrix, n_items)
  user_test_df = test_df[test_df['user_idx'] == uid]
  e = evaluate(user_series, train_df, user_test_df, K=10)
  all_scores[uid] = e

100%|██████████| 272184/272184 [22:44<00:00, 199.54it/s]


In [12]:
final_df = pd.DataFrame.from_dict(all_scores, orient='index').mean()
print("Item-CF Results:")
print(final_df)

Item-CF Results:
HR@10      0.244548
NDCG@10    0.174813
MRR@10     0.152236
dtype: float64


# Sample Recommendations

In [13]:
# Add game names for readability
idx_to_bggid = maps['idx2item']
games_lookup = games_df.set_index('BGGId')[['Name', 'AvgRating']]

def get_recommendations(user_idx, model, train_matrix, top_k=10):
  ids, scores = model.recommend(
        userid=user_idx,
        user_items=train_matrix[user_idx],
        N=top_k,
        filter_already_liked_items=True
    )
  # Map the model's internal indices to BGG IDs
  bgg_ids = [idx_to_bggid[i] for i in ids]

  # Retrieve the names from your games_lookup
  results = games_lookup.loc[bgg_ids].copy()
  results['Score'] = scores
  return results.reset_index()

# Sample 3 users
sample_users = test_df['user_idx'].sample(3, random_state=42).tolist()
for uid in sample_users:
    username = maps['idx2user'][uid]
    num_rated = len(train_df[train_df['user_idx'] == uid])
    print(f'\nTop 10 recommendations for user {username} | Rated {num_rated} games:')
    print(get_recommendations(uid, item_cf_model, train_matrix).to_string(index=False))


Top 10 recommendations for user VincentParadise | Rated 7 games:
 BGGId                                 Name  AvgRating    Score
 36218                             Dominion    7.61081 2.421464
 28143                  Race for the Galaxy    7.75771 2.214421
 68448                            7 Wonders    7.73733 2.081896
 30549                             Pandemic    7.59130 1.999129
 40692                          Small World    7.24834 1.985488
110327                   Lords of Waterdeep    7.74859 1.952221
  2651                           Power Grid    7.84328 1.919464
 37111 Battlestar Galactica: The Board Game    7.74169 1.912220
  3076                          Puerto Rico    7.96884 1.671340
 12333                    Twilight Struggle    8.28096 1.659325

Top 10 recommendations for user Anrab | Rated 7 games:
 BGGId                              Name  AvgRating    Score
167791                 Terraforming Mars    8.41879 1.056075
  2651                        Power Grid    7.84328 